In [1]:
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()

logging.langsmith("CH10-Retriever")

docs = [
    Document(
        page_content="수분 가득한 히알루론산 세럼으로 피부 속 깊은 곳까지 수분을 공급합니다.",
        metadata={"year": 2024, "category": "스킨케어", "user_rating": 4.7},
    ),
    Document(
        page_content="24시간 지속되는 매트한 피니시의 파운데이션, 모공을 커버하고 자연스러운 피부 표현이 가능합니다.",
        metadata={"year": 2023, "category": "메이크업", "user_rating": 4.5},
    ),
    Document(
        page_content="식물성 성분으로 만든 저자극 클렌징 오일, 메이크업과 노폐물을 부드럽게 제거합니다.",
        metadata={"year": 2023, "category": "클렌징", "user_rating": 4.8},
    ),
    Document(
        page_content="비타민 C 함유 브라이트닝 크림, 칙칙한 피부톤을 환하게 밝혀줍니다.",
        metadata={"year": 2023, "category": "스킨케어", "user_rating": 4.6},
    ),
    Document(
        page_content="롱래스팅 립스틱, 선명한 발색과 촉촉한 사용감으로 하루종일 편안하게 사용 가능합니다.",
        metadata={"year": 2024, "category": "메이크업", "user_rating": 4.4},
    ),
    Document(
        page_content="자외선 차단 기능이 있는 톤업 선크림, SPF50+/PA++++ 높은 자외선 차단 지수로 피부를 보호합니다.",
        metadata={"year": 2024, "category": "선케어", "user_rating": 4.9},
    ),
]

vectorstore = Chroma.from_documents(
    docs, OpenAIEmbeddings(model="text-embedding-3-small")
)

LangSmith 추적을 시작합니다.
[프로젝트명]
CH10-Retriever


In [2]:
from langchain_classic.chains.query_constructor.base import AttributeInfo

metadata_field_info = [
    AttributeInfo(
        name="category",
        description="The category of the cosmetic product. One of ['스킨케어', '메이크업', '클렌징', '선케어']",
        type="string",
    ),
    AttributeInfo(
        name="year",
        description="The year the cosmetic product was released",
        type="integer",
    ),
    AttributeInfo(
        name="user_rating",
        description="A user rating for the cosmetic product, ranging from 1 to 5",
        type="float",
    ),
]

In [3]:
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_community.query_constructors.chroma import ChromaTranslator
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model="gpt-4o-mini", temperature=0)

retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    document_contents="Brief summary of a cosmetic product",
    metadata_field_info=metadata_field_info,
    structured_query_translator=ChromaTranslator(),
)

d:\rag_one\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\user\AppData\Local\Temp\ipykernel_6128\2153908556.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.query_constructors.chroma import ChromaTranslator


In [4]:
retriever.invoke("평점이 4.8 이상인 제품을 추천해주세요.")
retriever.invoke("2023년에 출시된 상품을 추천해주세요.")
retriever.invoke("카테고리가 선케어인 상품을 추천해주세요.")

[Document(id='255675b1-5f15-4ca4-8298-b13d56a524c2', metadata={'user_rating': 4.9, 'category': '선케어', 'year': 2024}, page_content='자외선 차단 기능이 있는 톤업 선크림, SPF50+/PA++++ 높은 자외선 차단 지수로 피부를 보호합니다.')]

In [5]:
retriever.invoke(
    "카테고리가 메이크업인 상품 중에서 평점이 4.5 이상인 상품을 추천해주세요."
)

[Document(id='496bf1f7-8bdf-4b02-8763-8dd41ad089fd', metadata={'category': '메이크업', 'user_rating': 4.5, 'year': 2023}, page_content='24시간 지속되는 매트한 피니시의 파운데이션, 모공을 커버하고 자연스러운 피부 표현이 가능합니다.')]

In [7]:
retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    document_contents="Brief summary of a cosmetic product",
    metadata_field_info=metadata_field_info,
    enable_limit=True,
    search_kwargs={"k": 2},
    structured_query_translator=ChromaTranslator(),
)

In [8]:
retriever.invoke("2023년에 출시된 상품을 추천해주세요.")

[Document(id='496bf1f7-8bdf-4b02-8763-8dd41ad089fd', metadata={'year': 2023, 'user_rating': 4.5, 'category': '메이크업'}, page_content='24시간 지속되는 매트한 피니시의 파운데이션, 모공을 커버하고 자연스러운 피부 표현이 가능합니다.'),
 Document(id='d8b8bd61-e3e3-458a-b6e0-02ba83ff9780', metadata={'category': '스킨케어', 'user_rating': 4.6, 'year': 2023}, page_content='비타민 C 함유 브라이트닝 크림, 칙칙한 피부톤을 환하게 밝혀줍니다.')]

In [9]:
retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    document_contents="Brief summary of a cosmetic product",
    metadata_field_info=metadata_field_info,
    enable_limit=True,
    structured_query_translator=ChromaTranslator(),
)

retriever.invoke("2023년에 출시된 상품 1개를 추천해주세요.")
retriever.invoke("2023년에 출시된 상품 2개를 추천해주세요.")

[Document(id='496bf1f7-8bdf-4b02-8763-8dd41ad089fd', metadata={'year': 2023, 'category': '메이크업', 'user_rating': 4.5}, page_content='24시간 지속되는 매트한 피니시의 파운데이션, 모공을 커버하고 자연스러운 피부 표현이 가능합니다.'),
 Document(id='d8b8bd61-e3e3-458a-b6e0-02ba83ff9780', metadata={'category': '스킨케어', 'year': 2023, 'user_rating': 4.6}, page_content='비타민 C 함유 브라이트닝 크림, 칙칙한 피부톤을 환하게 밝혀줍니다.')]

#### 쿼리생성기 프롬프트를 체인에 연결하기


In [10]:
from langchain_classic.chains.query_constructor.base import(
    StructuredQueryOutputParser,
    get_query_constructor_prompt,
)

prompt = get_query_constructor_prompt(
    "Brief summary of a cosmetic product",
    metadata_field_info,
)

output_parser = StructuredQueryOutputParser.from_components()

query_constructor = prompt | llm | output_parser

In [ ]:
print(prompt.format(query="dummy question"))

TypeError: FewShotPromptTemplate.format() takes 1 positional argument but 2 were given